# 07. Quy mô toàn phần và so sánh tổng hợp

Chạy lại cấu hình tốt nhất của từng phương pháp ở $n = 1{,}2$ triệu. Chương 8.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

In [2]:
from src.dataset import load_processed, SWEEP, FULL
from src.experiment import BUILDERS, ExperimentGroup, run_group, summary_table
from src.figures import convergence_pair
title, build = BUILDERS["headline"]
out = {}
for tag, scale, folder in (("200k", SWEEP, "../results/raw"), ("1.2M", FULL, "../results/raw/full")):
    obj, *_ = load_processed("../data/processed", scale)
    out[tag] = run_group(ExperimentGroup("headline", title, build), obj, folder)
convergence_pair(out["200k"], "headline", title=title, out_dir="../results/figures")
convergence_pair(out["1.2M"], "headline-full", title=title + ", n = 1200000",
                 out_dir="../results/figures")

headline: loaded 5 runs from ../results/raw/headline.json
headline: loaded 5 runs from ../results/raw/full/headline.json


{'iters': [PosixPath('../results/figures/headline-full_iters.pdf'),
  PosixPath('../results/figures/headline-full_iters.png')],
 'time': [PosixPath('../results/figures/headline-full_time.pdf'),
  PosixPath('../results/figures/headline-full_time.png')]}

In [3]:
rows = {}
for tag, recs in out.items():
    for r in recs:
        rows.setdefault(r.label, {})[tag] = (r.iters[-1], round(r.total_time, 2))
pd.DataFrame([{"cấu hình": k, "vòng 200k": v["200k"][0], "vòng 1.2M": v["1.2M"][0],
               "giây 200k": v["200k"][1], "giây 1.2M": v["1.2M"][1],
               "tỷ lệ thời gian": round(v["1.2M"][1]/v["200k"][1], 2)}
              for k, v in rows.items()])

,cấu hình,vòng 200k,vòng 1.2M,giây 200k,giây 1.2M,tỷ lệ thời gian
0,GD (t = 1.9/L),2022,1959,27.52,158.67,5.77
1,"GD (Armijo c=0.3, rho=0.5, t0=1)",561,514,13.86,71.99,5.19
2,"AGD (beta from t, mu, restart)",286,283,3.92,23.49,5.99
3,"SGD (B = 2048, eta = 1/L_B)",3880,23400,2.02,13.17,6.52
4,"Newton (Hessian reused, t = 1)",1,1,0.03,0.16,5.33


Số vòng lặp gần như không đổi khi $n$ tăng 6 lần, vì $\kappa$ chỉ nhích từ 267,5
xuống 266,2. Thời gian tăng tuyến tính theo $n$, đúng mô hình chi phí $O(nd)$.

In [4]:
pd.DataFrame(summary_table(out["1.2M"]))

,method,configuration,status,iters_to_1e-06,seconds_to_1e-06,final_gap,iterations_run,total_seconds,fevals_per_iter,epochs
0,GD,t = 1.9/L,converged,337.0,26.922532,3.403689e-18,1959,158.672656,0.000000,NaN
1,GD,"Armijo (c = 0.3, rho = 0.5, t0 = 1)",stalled,144.0,16.572329,1.063898e-12,514,71.992291,4.521401,NaN
2,"AGD (beta from t, mu) + restart",t = 1/L,converged,74.0,6.128746,1.785489e-18,283,23.494197,0.000000,NaN
3,SGD (B = 2048),eta = 0.00309,max_iter,NaN,NaN,6.038311e-04,23400,13.166876,0.000000,39.936
4,Newton (Hessian reused),t = 1,converged,1.0,0.157841,0.000000e+00,1,0.157841,0.000000,NaN
